# Build a personal assistant with subagents

## Overview

The **supervisor pattern** is a [multi-agent](https://docs.langchain.com/oss/python/langchain/multi-agent) architecture where a central supervisor agent coordinates specialized worker agents.

When performance degrades: LLM makes too many errors in calling the right tool with the righ parameters.

Separate related **tools** and associated **prompts** into logical groups, to avoid **hallucination** caused by **context-rot**:

* A **calendar agent**, a box of [tools + instructions] for: scheduling, availability checking, and event management.
* An **email agent**, a box of [tools + instructions] for: drafting messages, and sending notifications.

The following figure shows context-rot: across models from various providers, performance still degrades even when their context-limit can theoretically "handle it":

![](../assets/context_rot.png){height=400}

### Understanding the architecture

![Architecture](../assets/subagents_arch_detailed.png)

Your system has three layers. The bottom layer contains rigid API tools that require exact formats. The middle layer contains sub-agents that accept natural language, translate it to structured API calls, and return natural language confirmations. The top layer contains the supervisor that routes to high-level capabilities and synthesizes results.

This separation of concerns provides several benefits: each layer has a focused responsibility, you can add new domains without affecting existing ones, and you can test and iterate on each layer independently.

### Concepts

We will cover the following concepts:

* [Multi-agent systems](/oss/python/langchain/multi-agent)

## Environment Setup

### Environment variables

The following environment variables are used in this notebook:

| Variable               | Required | Purpose                                  |
|------------------------|----------|------------------------------------------|
| `OPENROUTER_API_KEY`   | Yes      | Authenticates requests to OpenRouter LLMs |

#### On Colab?

Store these as Colab **Secrets** (key icon in the left sidebar), using the variable names above (for example `OPENROUTER_API_KEY`). Grant this notebook access to the secrets so they can be read securely at runtime.

#### Locally?

Store these in a `.env` file in your project directory (which should be listed in `.gitignore` so it is never committed). The code below uses `python-dotenv` to load values from this file into environment variables at runtime.


::: {.callout-warning}

API Keys are secrets and thus [**shall never be exposed**](./never_expose_api_keys.qmd).

:::

### Reading environment variables in

In [1]:
import os

In [2]:
try:
    # In Colab? read from userdata (secrets)
    from google.colab import userdata
    ON_COLAB = True
    os.environ["OPENROUTER_API_KEY"] = userdata.get("OPENROUTER_API_KEY")
except ImportError:
    # Load `.env` file (locally)
    from dotenv import load_dotenv
    load_dotenv(override=True)

### Install dependencies

In [3]:
# %pip install langchain langgraph langchain-community

## Select an LLM

Select a model that supports [tool-calling](https://docs.langchain.com/oss/python/integrations/providers/overview). Let's use the `nvidia/nemotron-3-nano-30b-a3b:free` model.

In [ ]:
from langchain_openai import ChatOpenAI


model_nemotron3_naon_omni = ChatOpenAI(
    model="nvidia/nemotron-3-nano-omni-30b-a3b-reasoning:free",
    temperature=0,
    # OpenRouter instead of the default OpenAI API
    base_url="https://openrouter.ai/api/v1",
    api_key=os.environ.get("OPENROUTER_API_KEY"),
)

## 1. Define tools

- Start by defining the tools that require structured inputs.
- In real applications, these would call actual APIs (Google Calendar, SendGrid, etc.).
- For this tutorial, you'll use stubs to demonstrate the pattern.

In [5]:
from pydantic import BaseModel, Field
from langchain.tools import tool

### General Tools

To provide the model with time-awareness, we'll define a `get_current_datetime()` tool:

In [27]:
from datetime import datetime

def get_current_datetime() -> str:
    """Return the current date and time in ISO 8601 format.
    Use this when the user refers to a date or time in a relative manner.
    Examples:
    - "in 2 days?
    - "next thursday?"
    - "next week at 2pm?"
    """
    return datetime.now().isoformat()

### Email Tools

In [6]:
class SendEmailInput(BaseModel):
    """Input schema for sending an email."""
    to: list[str] = Field(..., description="List of recipient email addresses")
    subject: str = Field(..., description="Subject of the email")
    body: str = Field(..., description="Body of the email message")
    cc: list[str] = Field(default=[], description="List of CC email addresses (optional)")

@tool(args_schema=SendEmailInput)
def send_email(
    to: list[str],
    subject: str,
    body: str,
    cc: list[str] = []
) -> str:
    """Send an email via email API. Requires properly formatted addresses."""
    # Stub: In practice, this would call SendGrid, Gmail API, etc.
    return f"Email sent to {', '.join(to)} - Subject: {subject}"


### Calendar Tools

In [7]:
class CalendarEventInput(BaseModel):
    """Input schema for creating a calendar event."""
    title: str = Field(description="Title of the event")
    start_time: str = Field(description="Start time in ISO datetime format")
    end_time: str = Field(description="End time in ISO datetime format")
    attendees: list[str] = Field(description="List of attendee email addresses")
    location: str = Field(default="", description="Event location (optional)")

@tool(args_schema=CalendarEventInput)
def create_calendar_event(
    title: str,
    start_time: str,
    end_time: str,
    attendees: list[str],
    location: str = ""
) -> str:
    """Create a calendar event. Requires exact ISO datetime format."""
    # Stub: In practice, this would call Google Calendar API, Outlook API, etc.
    return f"Event created: {title} from {start_time} to {end_time} with {len(attendees)} attendees"

In [8]:
class AvailableTimeSlotsInput(BaseModel):
    """Input schema for checking available time slots."""
    attendees: list[str] = Field(..., description="List of attendee email addresses")
    date: str = Field(..., description="Date to check availability (ISO format: '2024-01-15')")
    duration_minutes: int = Field(..., description="Duration for the meeting in minutes")

@tool(args_schema=AvailableTimeSlotsInput)
def get_available_time_slots(
    attendees: list[str],
    date: str,
    duration_minutes: int
) -> list[str]:
    """Check calendar availability for given attendees on a specific date."""
    # Stub: In practice, this would query calendar APIs
    return ["09:00", "14:00", "16:00"]

## 2. Create each sub-agent individually

### Create an email agent

- The email agent handles message composition and sending.
- It focuses on extracting recipient information, crafting appropriate subject lines and body text, and managing email communication.

In [38]:
from langchain.agents import create_agent
from langgraph.checkpoint.memory import InMemorySaver

checkpointer_email = InMemorySaver()

EMAIL_AGENT_PROMPT = (
    "You are an email assistant. "
    "Compose professional emails based on natural language requests. "
    "Extract recipient information and craft appropriate subject lines and body text. "
    "Use send_email to send the message. "
    "Always confirm what was sent in your final response."
)

email_agent = create_agent(
    model=model_nemotron3_naon_omni,
    tools=[
        send_email,
        get_current_datetime,
    ],
    system_prompt=EMAIL_AGENT_PROMPT,
    checkpointer=checkpointer_email,
)

Test the email agent with a natural language request:

In [40]:
from langchain.messages import HumanMessage

config = {"configurable": {"thread_id": 1}}
prompt1 = "Send charlie@example.com a reminder about reviewing the new mockups"
for step in email_agent.stream(
    input={"messages": [HumanMessage(prompt1)]},
    config=config,
    stream_mode="values"
):
    step['messages'][-1].pretty_print()

================================ Human Message =================================

Send charlie@example.com a reminder about reviewing the new mockups
================================== Ai Message ==================================
Tool Calls:
  send_email (call_bd03ba392f634a3e89184614)
 Call ID: call_bd03ba392f634a3e89184614
  Args:
    to: ['charlie@example.com']
    subject: Reminder: Review New Mockups
    body: Hi Charlie,

This is a friendly reminder to review the new mockups. Please take a look and provide your feedback at your earliest convenience.

Best regards,
[Your Name]
================================= Tool Message =================================
Name: send_email

Email sent to charlie@example.com - Subject: Reminder: Review New Mockups
================================== Ai Message ==================================

The emailhas been successfully sent to charlie@example.com with the subject "Reminder: Review New Mockups". 

<final_response>
Email sent to charlie@exampl

- The agent infers the recipient from the informal request, crafts a professional subject line and body, calls `send_email`, and returns a confirmation.
- Each sub-agent has a narrow focus with domain-specific tools and prompts, allowing it to excel at its specific task.

### Create a calendar agent

- The calendar agent understands natural language scheduling requests and translates them into precise API calls.
- It handles date parsing, availability checking, and event creation.

In [42]:
from langchain.agents import create_agent
from langgraph.checkpoint.memory import InMemorySaver

checkpointer_calendar = InMemorySaver()

CALENDAR_AGENT_PROMPT = (
    "You are a calendar scheduling assistant. "
    "Parse natural language scheduling requests (e.g., 'next Tuesday at 2pm') "
    "into proper ISO datetime formats. "
    "Use get_available_time_slots to check availability when needed. "
    "Use create_calendar_event to schedule events. "
    "Always confirm what was scheduled in your final response."
)

calendar_agent = create_agent(
    model=model_nemotron3_naon_omni,
    tools=[
        create_calendar_event,
        get_available_time_slots,
        get_current_datetime,
    ],
    system_prompt=CALENDAR_AGENT_PROMPT,
    checkpointer=checkpointer_calendar,
)

Test the calendar agent to see how it handles natural language scheduling:

In [43]:
from langchain.messages import HumanMessage

config = {"configurable": {"thread_id": 1}}

prompt = "Schedule a team meeting next Tuesday at 2pm for 1 hour with Alan, Bob and Charlie"
for step in calendar_agent.stream(
    input={"messages": [HumanMessage(prompt)]},
    config=config,
    stream_mode="values"
):
    step['messages'][-1].pretty_print()

================================ Human Message =================================

Schedule a team meeting next Tuesday at 2pm for 1 hour with Alan, Bob and Charlie
================================== Ai Message ==================================
Tool Calls:
  get_current_datetime (chatcmpl-tool-a0bec9601e42660a)
 Call ID: chatcmpl-tool-a0bec9601e42660a
  Args:
================================= Tool Message =================================
Name: get_current_datetime

2026-05-01T08:35:55.205644
================================== Ai Message ==================================

I need the email addresses for Alan, Bob, and Charlie to schedule the meeting. Could you please provide their email addresses?


In [44]:
prompt = "their emails are: alan@example.com, bob@example.com, and charlie@example.com"

for step in calendar_agent.stream(
    input={"messages": [HumanMessage(prompt)]},
    config=config,
    stream_mode="values"
):
    step['messages'][-1].pretty_print()

================================ Human Message =================================

their emails are: alan@example.com, bob@example.com, and charlie@example.com
================================== Ai Message ==================================
Tool Calls:
  get_available_time_slots (call_0640a8d9a5f24bc99931c56f)
 Call ID: call_0640a8d9a5f24bc99931c56f
  Args:
    attendees: ['alan@example.com', 'bob@example.com', 'charlie@example.com']
    date: 2026-05-06
    duration_minutes: 60
================================= Tool Message =================================
Name: get_available_time_slots

["09:00", "14:00", "16:00"]
================================== Ai Message ==================================
Tool Calls:
  create_calendar_event (call_a21c2e97358e4503a5b64fa7)
 Call ID: call_a21c2e97358e4503a5b64fa7
  Args:
    end_time: 2026-05-06T15:00:00
    start_time: 2026-05-06T14:00:00
    attendees: ['alan@example.com', 'bob@example.com', 'charlie@example.com']
    title: Team Meeting
=======

The agent parses "next Tuesday at 2pm" into ISO format ("2024-01-16T14:00:00"), calculates the end time, calls `create_calendar_event`, and returns a natural language confirmation.

## 3. Wrap sub-agents as tools

- Now wrap each sub-agent as a tool that the supervisor can invoke.
- This is the key architectural step that creates the layered system.
- The supervisor will see high-level tools like `"schedule_event"`, not low-level tools like `"create_calendar_event"`.

In [45]:
@tool
def schedule_event(request: str) -> str:
    """Schedule calendar events using natural language.

    Use this when the user wants to create, modify, or check calendar appointments.
    Handles date/time parsing, availability checking, and event creation.

    Input: Natural language scheduling request (e.g., 'meeting with design team
    next Tuesday at 2pm')
    """
    result = calendar_agent.invoke({"messages": [HumanMessage(request)]})
    return result["messages"][-1].text

In [46]:
@tool
def manage_email(request: str) -> str:
    """Send emails using natural language.

    Use this when the user wants to send notifications, reminders, or any email
    communication. Handles recipient extraction, subject generation, and email
    composition.

    Input: Natural language email request (e.g., 'send them a reminder about
    the meeting')
    """
    result = email_agent.invoke({"messages": [HumanMessage(request)]})
    return result["messages"][-1].text


- The **tool descriptions (docstring)** help the supervisor decide when to use each tool, so make them clear and specific.
- We return only the sub-agent's final response, as the supervisor doesn't need to see intermediate reasoning or tool calls.

## 4. Create the supervisor agent

- Now create the supervisor that orchestrates the sub-agents.
- The supervisor only sees high-level tools and makes routing decisions at the domain level, not the individual API level.

In [49]:
checkpointer_supervisor = InMemorySaver()

SUPERVISOR_PROMPT = (
    "You are a helpful personal assistant. "
    "You can schedule calendar events and send emails. "
    "Break down user requests into appropriate tool calls and coordinate the results. "
    "When a request involves multiple actions, use multiple tools in sequence."
)

supervisor_agent = create_agent(
    model=model_nemotron3_naon_omni,
    tools=[
        schedule_event,
        manage_email
    ],
    system_prompt=SUPERVISOR_PROMPT,
    checkpointer=checkpointer_supervisor,
)

## 5. Use the supervisor

Now test your complete system with complex requests that require coordination across multiple domains:


### Example 1: Simple single-domain request

In [50]:
config = {"configurable": {"thread_id": 1}}
prompt = "Schedule a team standup for tomorrow at 9am"
for step in supervisor_agent.stream(
    input={"messages": [HumanMessage(prompt)]},
    config=config,
    stream_mode="values"
):
    step['messages'][-1].pretty_print()

================================ Human Message =================================

Schedule a team standup for tomorrow at 9am
================================== Ai Message ==================================
Tool Calls:
  schedule_event (call_34b2584fa75f4544974205e9)
 Call ID: call_34b2584fa75f4544974205e9
  Args:
    request: Schedule a team standup for tomorrow at 9am
================================= Tool Message =================================
Name: schedule_event

To schedule theteam standup for tomorrow at 9am, I need the email addresses of the attendees. Could you please provide the list of team members' email addresses to include in the event?
================================== Ai Message ==================================

Please provide the list of team members' email addresses to include in the event.


The supervisor identifies this as a calendar task, calls `schedule_event`, and the calendar agent handles date parsing and event creation.

### Example 2: Complex multi-domain request

In [51]:
config = {"configurable": {"thread_id": 2}}
prompt = (
    "Schedule a team meeting next Tuesday at 2pm for 1 hour with Alan, Bob and Charlie "
    "their emails are: alan@example.com, bob@example.com, and charlie@example.com"
    "and send Charlie a reminder about reviewing the new mockups"
)
for step in supervisor_agent.stream(
    input={"messages": [HumanMessage(prompt)]},
    config=config,
    stream_mode="values"
):
    step['messages'][-1].pretty_print()

================================ Human Message =================================

Schedule a team meeting next Tuesday at 2pm for 1 hour with Alan, Bob and Charlie their emails are: alan@example.com, bob@example.com, and charlie@example.comand send Charlie a reminder about reviewing the new mockups
================================== Ai Message ==================================
Tool Calls:
  schedule_event (call_dfdd56a39cca4e6ab38cba01)
 Call ID: call_dfdd56a39cca4e6ab38cba01
  Args:
    request: Schedule a team meeting next Tuesday at 2pm for 1 hour with Alan, Bob, and Charlie (alan@example.com, bob@example.com, charlie@example.com)
  manage_email (call_efbee7e312404780a9644a9b)
 Call ID: call_efbee7e312404780a9644a9b
  Args:
    request: send Charlie a reminder about reviewing the new mockups
================================= Tool Message =================================
Name: manage_email

Ineed Charlie's email address to send the reminder. Could you please provide his email addre

- The supervisor recognizes this requires both calendar and email actions, calls `schedule_event` for the meeting, then calls `manage_email` for the reminder.
- Each sub-agent completes its task, and the supervisor synthesizes both results into a coherent response.

## Your Turn

Try adding this tool to the mix, such that the user doesn't have to specify the contact emails, the AI should simply know which emails belong to Alice, Bob and Charlie, by using this lookup tool:

In [ ]:
# !uv add rapidfuzz
# %pip install rapidfuzz
from rapidfuzz import process, fuzz

class ContactLookupInput(BaseModel):
    """Input schema for contact lookup."""
    name: str = Field(..., description="Name to search for in contacts (partial or fuzzy match allowed)")

# Hardcoded contacts for demonstration
CONTACTS_DB = {
    "Alice Johnson": {"email": "alice@example.com"},
    "Bob Smith": {"email": "bob.smith@example.com"},
    "Charlie Kim": {"email": "charlie.kim@example.com"},
    "Dana Lee": {"email": "dana.lee@example.com"},
    "Evan Wright": {"email": "evan.wright@example.com"},
}

@tool(args_schema=ContactLookupInput)
def lookup_contact(name: str) -> dict:
    """Lookup contact information by (possibly partial or fuzzy) name.
    Returns the best matching contact's email (and name).
    """
    if not name:
        return {"error": "No name provided."}
    choices = list(CONTACTS_DB.keys())
    best_match, score, _ = process.extractOne(
        name, choices, scorer=fuzz.WRatio, score_cutoff=60
    )
    if best_match:
        info = CONTACTS_DB[best_match]
        return {"matched_name": best_match, "email": info["email"], "score": score}
    else:
        return {"error": f"No contact found matching '{name}'."}

In [ ]:
# YOUR CODE 
# Step 1. Add the lookup_contact tool to the supervisor agent
# Step 2. Test the system with a request that requires contact lookup
# Step 3. Run the agent without mentioning the contact emails explicitly to test the whole workflow

## Key takeaways

- The supervisor pattern creates layers of abstraction where each layer has a clear responsibility.
- When designing a supervisor system, start with clear domain boundaries and give each sub-agent focused tools and prompts.
- Write clear tool descriptions for the supervisor, test each layer independently before integration, and control information flow based on your specific needs.